## Foundation for script for automatic data generation 1H, 13C, HSQC, COSY NMR
- Master in notebook 20.1

### What this notebook does

This notebook is the **synthetic-data generator** that sits between the release and the models: it takes a CSV of SMILES and produces simulated 1H, 13C, COSY and HSQC spectra, which the `1.0_Experiment_Notebook___` then consumes as training/validation input.

Pipeline (all real work happens in [utils_MMT/data_generation_v15_4.py](file:///d:/Challenge/MultiModalSpectralTransformer/utils_MMT/data_generation_v15_4.py)):

1. **`run_sgnn`** — a graph neural network predicts 13C and 1H chemical shifts for every molecule, writing SDF files. Molecules are filtered first: must contain H, must be connected, molecular weight must be <= `SGNN_size_filter`.
2. **`run_1H_generation`** — turns each shift into a spectrum, including J-coupling multiplets built from neighbouring hydrogens.
3. **`run_13C_generation`** — same for carbon, with symmetry-equivalent atoms consolidated into one peak.
4. **`run_COSY_generation`** — 2D cross peaks between coupling partners.
5. **`run_HSQC_generation`** — 2D one-bond 1H–13C correlations.
6. **`cleanup_files`** — moves the intermediate SDF files into a dated sub-folder.

Three cells:

| Cell | Role |
|---|---|
| 1 | Loads the four token vocabularies and creates a run id |
| 2 | Builds a `Namespace` config (all options are one-element lists, argparse convention) |
| 3 | Runs the pipeline and prints the shape of every produced array |

Run it from the repository root, otherwise the relative `itos.json` paths fail.

In [ ]:
# ---------------------------------------------------------------------------
# Third-party imports used by this notebook.
# The notebook itself only wires things together; the real NMR simulation
# lives in utils_MMT/data_generation_v15_4.py.
# ---------------------------------------------------------------------------
import wandb                        # experiment tracking (unused here, kept for parity with 1.0)
import matplotlib.pyplot as plt     # plotting used indirectly by the library's spectrum functions
import numpy as np                  # every generated spectrum comes back as a np.ndarray
import pandas as pd                 # tables: SMILES in, generated spectra rows out
from rdkit import Chem              # cheminformatics toolkit: SMILES -> molecule graph
from rdkit.Chem import Draw, MolFromSmiles, MolToSmiles  # drawing / SMILES <-> Mol conversion
from tqdm import tqdm               # progress bars inside the long simulation loops
from rdkit import DataStructs       # fingerprints, used by similarity helpers
import json                         # reading the four token vocabularies
import torch                        # only needed by the SGNN model that predicts shifts
import os                           # path handling
import random                       # generates the run id (rand_num)

# Type aliases used by the annotations below (PEP 484).
from typing import Any, Dict, List, Tuple

# Vocabulary shapes (see lesson 1):
#   stoi     : token text -> int id      (43 entries, SMILES characters)
#   itos     : str(id)    -> token text  (reverse of stoi)
#   stoi_MF  : token text -> int id      (212 entries, molecular-formula tokens)
#   itos_MF  : str(id)    -> token text  (reverse of stoi_MF)


def load_json_dics() -> Tuple[Dict[str, str], Dict[str, int], Dict[str, int], Dict[str, str]]:
    """Load the four token vocabularies from the current working directory.

    The notebook must be started from the repository root (the folder that
    contains ``itos.json``), because every path below is relative to
    ``os.getcwd()``.

    Returns:
        Tuple ``(itos, stoi, stoi_MF, itos_MF)`` where

        * ``itos``    -- ``{"0": "<PAD>", ..., "42": "B"}``    (id -> SMILES token)
        * ``stoi``    -- ``{"<PAD>": 0, ..., "B": 42}``        (SMILES token -> id)
        * ``stoi_MF`` -- ``{"<SOS>": 1, ..., "100": 212}``     (formula token -> id)
        * ``itos_MF`` -- reverse of ``stoi_MF``

    Example:
        >>> itos, stoi, stoi_MF, itos_MF = load_json_dics()
        >>> len(stoi), len(stoi_MF)
        (43, 212)
        >>> stoi['C'], itos['5']
        (5, 'C')

    Note:
        ``itos`` keys are strings (JSON object keys must be strings), so
        downstream code always indexes it as ``itos[str(int(id))]``.
    """
    # Anchor every lookup to the notebook's start directory.
    current_dir: str = os.getcwd()

    # SMILES character vocabulary, id -> token; used when decoding generated strings.
    with open(os.path.join(current_dir, 'itos.json'), 'r') as f:
        itos: Dict[str, str] = json.load(f)

    # SMILES character vocabulary, token -> id; used when encoding training data.
    with open(os.path.join(current_dir, 'stoi.json'), 'r') as f:
        stoi: Dict[str, int] = json.load(f)

    # Molecular-formula vocabulary: tokens are element symbols and counts ("C", "6").
    with open(os.path.join(current_dir, 'stoi_MF.json'), 'r') as f:
        stoi_MF: Dict[str, int] = json.load(f)

    with open(os.path.join(current_dir, 'itos_MF.json'), 'r') as f:
        itos_MF: Dict[str, str] = json.load(f)

    return itos, stoi, stoi_MF, itos_MF


# Declare the types once, then assign. Keeps the names readable in every cell.
itos: Dict[str, str]
stoi: Dict[str, int]
stoi_MF: Dict[str, int]
itos_MF: Dict[str, str]
itos, stoi, stoi_MF, itos_MF = load_json_dics()

# Unique id of this run, later used to name output files/folders so that two
# runs never overwrite each other. Example value: "4820193".
rand_num: str = str(random.randint(1, 10000000))

In [ ]:
# Imports used by this cell, kept here so the cell stays runnable on its own.
from argparse import Namespace   # dict-like object whose keys become attributes (config.SGNN_size_filter)
import pandas as pd              # re-imported for parity with the original cell; pd is used in cell 3
import torch                     # ditto: the pipeline needs torch, imported here as before
from torch.utils.data import DataLoader  # used by the library's dataloaders, imported for parity

# The repository root is assumed to be the notebook's start directory.
# Every path below is built from it, so the notebook is portable as long as it
# is launched from the repo root (JupyterLab "Open from path").
base_dir: str = os.getcwd()

# ---------------------------------------------------------------------------
# All hyperparameters are stored as *lists* holding exactly one element.
# That convention comes from the original argparse design, where each option
# was parsed as a list; parse_arguments() below unwraps the single element:
#     {"SGNN_size_filter": [550]}  ->  config.SGNN_size_filter == 550
# ---------------------------------------------------------------------------
hyperparameters: Dict[str, List[Any]] = {
    # Run identifier; comes from cell 1 and is copied into config for logging.
    "ran_num": [rand_num],

    # Token vocabularies consumed by the SGNN encoder/decoder: one for SMILES,
    # one for molecular formulae (the files explained in lesson 1).
    "itos_path": [os.path.join(base_dir, "itos.json")],
    "stoi_path": [os.path.join(base_dir, "stoi.json")],
    "itos_MF_path": [os.path.join(base_dir, "itos_MF.json")],
    "stoi_MF_path": [os.path.join(base_dir, "stoi_MF.json")],

    # Folder that receives the CSV files describing the generated molecules.
    "gen_mol_csv_folder_path": ["./data/SGNN_gen_folder"],

    # ---- SGNN -> simulated 1H / 13C / HSQC / COSY -------------------------
    # Working folder of the SGNN step: SDF files with predicted shifts land here.
    "SGNN_gen_folder_path": [os.path.join(base_dir, "experiments/SGNN_gen_folder")],

    # Input SMILES list. The release ships a 20-molecule CSV used as a smoke test;
    # point this at a bigger CSV to generate a real dataset.
    "SGNN_csv_gen_smi": [os.path.join(base_dir, "data/test_data/IBM_SMI_data_top20.csv")],

    # Molecular-weight cutoff in Da. Molecules heavier than this are dropped
    # before SGNN inference (data_generation_v15_4.py:84).
    # Example: aspirin C9H8O4 = 180.16 Da -> kept; a 600 Da peptide -> dropped.
    "SGNN_size_filter": [550],

    # Where the per-modality CSVs (1H / 13C / COSY / HSQC) are written.
    "SGNN_csv_save_folder": [os.path.join(base_dir, "./data/SGNN_gen_folder")],

    # Where simulated IR spectra would be written. Unused in this notebook,
    # kept for parity with the scripts/*.py entry points.
    "IR_save_folder": [os.path.join(base_dir, "./data/SGNN_gen_folder/IR_data")],
}


def parse_arguments(hyperparameters: Dict[str, List[Any]]) -> Namespace:
    """Unwrap ``{name: [value]}`` into an attribute-style namespace.

    Why lists: the original code base used ``argparse`` with ``nargs``, so every
    option is a one-element list. This helper unwraps it so the rest of the
    pipeline can write ``config.SGNN_size_filter`` instead of
    ``config.SGNN_size_filter[0]``.

    Args:
        hyperparameters: Mapping of option name to a single-element list,
            e.g. ``{"SGNN_size_filter": [550]}``.

    Returns:
        An ``argparse.Namespace`` whose attributes are the unwrapped values.

    Example:
        >>> config = parse_arguments({"SGNN_size_filter": [550]})
        >>> config.SGNN_size_filter
        550
        >>> hasattr(config, "missing_option")
        False

    Note:
        Values are shared with ``hyperparameters`` (no deep copy), so mutating
        a mutable value later would affect both objects.
    """
    # ("SGNN_size_filter", [550]) -> ("SGNN_size_filter", 550)
    parsed_args: Dict[str, Any] = {key: val[0] for key, val in hyperparameters.items()}
    # **parsed_args expands the dict into keyword arguments,
    # i.e. Namespace(SGNN_size_filter=550, ran_num="4820193", ...)
    return Namespace(**parsed_args)


config: Namespace = parse_arguments(hyperparameters)

In [ ]:
# The whole pipeline is implemented in one library module; the notebook only calls it.
from utils_MMT.data_generation_v15_4 import run_sgnn, main_run_data_generation

# Make sure the CSV output folder exists before the pipeline writes into it.
# exist_ok=True keeps re-runs idempotent (no FileExistsError on the 2nd run).
import os
os.makedirs("./data/SGNN_gen_folder", exist_ok=True)

# Run the five-stage pipeline (data_generation_v15_4.py:1282):
#   1. run_sgnn            -> SGNN predicts 13C / 1H shifts and writes SDF files
#   2. run_1H_generation   -> simulated 1H spectra incl. J-coupling multiplets
#   3. run_13C_generation  -> simulated 13C spectra, symmetry-consolidated
#   4. run_COSY_generation -> 2D COSY cross peaks
#   5. run_HSQC_generation -> 2D HSQC cross peaks
#
# Stage 1 is by far the most expensive: it runs a graph neural network over every
# molecule that survives the three filters applied in run_sgnn():
#   - must contain at least one hydrogen
#   - must be a connected structure (no ".")
#   - molecular weight <= config.SGNN_size_filter (550 Da)
# With the default 20-molecule CSV this finishes quickly; with a 450k-row CSV it
# takes hours, so keep an eye on the kernel.
combined_df: pd.DataFrame
data_1H: np.ndarray
data_13C: np.ndarray
data_COSY: np.ndarray
data_HSQC: np.ndarray
csv_1H_path: str
csv_13C_path: str
csv_COSY_path: str
csv_HSQC_path: str
combined_df, data_1H, data_13C, data_COSY, data_HSQC, csv_1H_path, csv_13C_path, csv_COSY_path, csv_HSQC_path = main_run_data_generation(config)

# ---------------------------------------------------------------------------
# Sanity checks. Each *_shape is (number of generated spectra, feature width);
# the exact widths are dictated by the 1.0 notebook's dataloaders, so print them
# once and compare against a known-good run after any code change.
# ---------------------------------------------------------------------------
# 1H dataset: one row per molecule, chemical shifts + split patterns.
print(f"1H data shape: {data_1H.shape}")

# 13C dataset: one row per molecule, consolidated carbon shifts.
print(f"13C data shape: {data_13C.shape}")

# COSY dataset: 2D correlation peaks as sparse (x, y) pairs.
print(f"COSY data shape: {data_COSY.shape}")

# HSQC dataset: one-bond 1H-13C correlations as (1H, 13C) pairs.
print(f"HSQC data shape: {data_HSQC.shape}")

# Where the four CSVs were written; the training notebook reads these paths.
print(f"1H CSV saved to: {csv_1H_path}")
print(f"13C CSV saved to: {csv_13C_path}")
print(f"COSY CSV saved to: {csv_COSY_path}")
print(f"HSQC CSV saved to: {csv_HSQC_path}")